# ---------- Imports ----------

In [1]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, plot_confusion_matrix, confusion_matrix
from skmultiflow.meta import AdaptiveRandomForestClassifier
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import plot_tree
from tqdm import tqdm
import joblib
from math import ceil
from scipy.signal import medfilt
from scipy.optimize import curve_fit
from river import metrics
from river import ensemble
import os
import pandas as pd
import seaborn as sns
from joblib import Parallel, delayed
import time
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import medfilt
from scipy.optimize import curve_fit
import math
from scipy.signal import find_peaks
import pandas as pd
from scipy import interpolate
from tqdm import tqdm
from scipy.signal import find_peaks
import math
import csv
import pandas as pd
import os
from sklearn.utils import shuffle
from collections import Counter


In [2]:
# Define the directory paths
training_direct = #folder containing the data to be used to train
labelled_direct =  #folder containing the training data thats been labelled
savemodel_file =  #file for the saved model
savemodel_direct = #directory to save the model
figsaving_direct = #folder containing saved figures

# ---------- Workhorse Functions ----------

In [3]:
def deltanu_calc(numax):
    #calculates delta nu upper and lower bound
    #uses frequency units eg Hz
    #returns in frequency units
    deltanu = 0.263*numax**0.772
    deltanumin = ((0.263-0.009) *numax**(0.772-0.005))
    deltanumax = ((0.263+0.009) *numax**(0.772+0.005))
    return deltanu, deltanumin, deltanumax


def peak_check(first_peak_height, search_region, threshold):
    #this searches for the peak in the given region, returns True only if a peak exists
#search region is just the section of the power array we care about, eg search_region=pnoderegionpower[0:50]
    if not np.any(search_region):
            search_region = 1

    peak_height = np.max(search_region)



    peak_position_index = np.argmax(search_region)
    peak_above_threshold = peak_height-threshold
    if peak_above_threshold<0:
        return False, peak_position_index, 0, 0
    ratio = peak_height/first_peak_height

    if ratio<0.3: #originally tried with 0.75
        return False, ratio, peak_height, peak_position_index
    return True, ratio, peak_height, peak_position_index

def search_consecutive_peak(first_peak_position_index, threshold, reverse=False, plotting=False,numax=False):
    first_peak_height = pnoderegionpower[first_peak_position_index]
    guess_numax_hz = pnoderegionfrequency[first_peak_position_index]
    #print(guess_numax_hz)
    if numax:
        guess_numax_hz = numax
    deltanu, deltanumin, deltanumax = deltanu_calc(guess_numax_hz)
    #print(guess_numax_hz)
    #print(first_peak_position_index,first_peak_height)
    deltanu = int((deltanu/conversion))
    deltanumin = int(np.floor((deltanumin/conversion)))
    deltanumax = int(np.ceil((deltanumax/conversion)))
    result=[]
    peak_position_index =[]

    if plotting:
        plt.plot(pnoderegionfrequency, pnoderegionpower)

    for i in range(20):
        if reverse==False:
            lowerbound = first_peak_position_index+(deltanu)*i + deltanumin
            upperbound= first_peak_position_index+(deltanu)*i + deltanumax
            if lowerbound == upperbound:
                upperbound+=1

            if plotting:
                plt.axvline(pnoderegionfrequency[lowerbound], color='red')
                plt.axvline(pnoderegionfrequency[upperbound],color='red')

        else:
            upperbound = first_peak_position_index-(deltanu)*i - deltanumin
            lowerbound= first_peak_position_index-(deltanu)*i - deltanumax
            if lowerbound == upperbound:
                upperbound+=1
            if plotting:
                plt.axvline(pnoderegionfrequency[lowerbound], color='red')
                plt.axvline(pnoderegionfrequency[upperbound],color='red')


        search_region=pnoderegionpower[lowerbound:upperbound]

        if not np.any(search_region):
            break

        check_for_peak = peak_check(first_peak_height,search_region,threshold)
        result.append(check_for_peak[0])
        if check_for_peak[0] ==True:
            peak_position_index.append(check_for_peak[3]+lowerbound)
            #print(f'Peak at: {(check_for_peak[3]+lowerbound+pnoderegionfrequency.shape[0])*conversion}')
        if check_for_peak[0] == False:
            #print('No More Peaks')
            break



    return result, peak_position_index




def find_initial_peak():
    #this will be used as a guess of numax
    index_tallest_peak = np.argmax(pnoderegionpower)
    height_tallest_peak = pnoderegionpower[index_tallest_peak]
    return index_tallest_peak


def find_best_numax_from_one_peak(peak_overtone1_original, threshold):
    peak1_height = pnoderegionpower[peak_overtone1_original]
    guess_numax = pnoderegionfrequency[peak_overtone1_original]
    results_num_peaks=[]
    peaks= []
    numax_iterations=[]
    starting_peak_changed =[]
    for i in range(21):
        guess_numax=pnoderegionfrequency[peak_overtone1_original-10+i]
        #print(guess_numax)
        for i in range(5):
            peaks_i = []
            peak_overtone1 = peak_overtone1_original-2+i
            starting_peak_changed.append(peak_overtone1)
            #print(peak_overtone1)
            search1= search_consecutive_peak(peak_overtone1, threshold,  numax=guess_numax)
            search1left= search_consecutive_peak(peak_overtone1, threshold, reverse=True,numax=guess_numax)
            #print(search1)
            num_peaks = search1[
                0].count(True) +search1left[
                0].count(True)+1 #the plus 1 is from the 1 peaks we input into the function

            peaks_i.append(search1[1])
            peaks_i.append(search1left[1])

            peaks.append(peaks_i)
            results_num_peaks.append(num_peaks)
            numax_iterations.append(guess_numax)



    best_result_num_peaks = np.max(results_num_peaks)
    best_result_num_peaks_index= np.argmax(results_num_peaks)
    best_peak_locations = peaks[best_result_num_peaks_index]
    best_numax = numax_iterations[best_result_num_peaks_index]
    best_starting_peak = starting_peak_changed[best_result_num_peaks_index]

    return best_result_num_peaks, best_peak_locations, best_numax,  best_starting_peak


def find_other_overtone_near_numax(peak, secondpeak, threshold):
    peak_height = pnoderegionpower[peak]
    if peak> secondpeak:
        upperbound = peak-1
        lowerbound= secondpeak+1
    else:
        upperbound = secondpeak-1
        lowerbound= peak+1

    search_region = pnoderegionpower[lowerbound:upperbound]


    check_for_peak = peak_check(peak_height,search_region,threshold)

    other_overtone_peak_position = check_for_peak[3]+lowerbound
    return other_overtone_peak_position



def find_best_numax_from_both_peak(peak_overtone1_original, peak_overtone2, threshold):
    guess_numax = pnoderegionfrequency[peak_overtone1_original]
    results_num_peaks=[]
    peaks= []
    numax_iterations=[]
    starting_peak_changed =[]
    for i in range(21):
        guess_numax=pnoderegionfrequency[peak_overtone1_original-10+i]
        #print(guess_numax)
        for i in range(5):
            peaks_i = []
            peak_overtone1 = peak_overtone1_original-2+i
            starting_peak_changed.append(peak_overtone1)
            #print(peak_overtone1)
            peak2_height=pnoderegionpower[peak_overtone2]
            peak1_height = pnoderegionpower[peak_overtone1_original]
            search1= search_consecutive_peak(peak_overtone1, threshold, numax=guess_numax)
            search1left= search_consecutive_peak(peak_overtone1, threshold, reverse=True,numax=guess_numax)
            search2 = search_consecutive_peak(peak_overtone2, threshold, numax=guess_numax)
            search2left= search_consecutive_peak(peak_overtone2, threshold, reverse=True,numax=guess_numax)
            #print(search1)
            num_peaks = search1[
                0].count(True) +search1left[
                0].count(True)+search2[
                0].count(True)+search2left[
                0].count(True)+2 #the plus 2 is from the 2 peaks we input into the function

            peaks_i.append(search1[1])
            peaks_i.append(search1left[1])
            peaks_i.append(search2[1])
            peaks_i.append(search2left[1])


            peaks.append(peaks_i)
            results_num_peaks.append(num_peaks)
            numax_iterations.append(guess_numax)



    best_result_num_peaks = np.max(results_num_peaks)
    best_result_num_peaks_index= np.argmax(results_num_peaks)
    best_peak_locations = peaks[best_result_num_peaks_index]
    best_numax = numax_iterations[best_result_num_peaks_index]
    best_starting_peak = starting_peak_changed[best_result_num_peaks_index]

    return best_result_num_peaks, best_peak_locations, best_numax,  best_starting_peak

def grand_function(binned_f, binned_p,binsize,threshold):

    global pnoderegionfrequency
    global pnoderegionpower
    global white_noise
    global conversion
    pnoderegionfrequency, pnoderegionpower, white_noise, pmode_indices = findpregion(
        binsize, binned_p, binned_f)
    conversion =(np.max(pnoderegionfrequency)-np.min(pnoderegionfrequency))/pnoderegionfrequency.shape[0]
    x=find_initial_peak()


    tallest_peak = find_initial_peak()
    best_result_num_peaks, best_peak_locations, best_numax,  best_starting_peak =find_best_numax_from_one_peak(
        tallest_peak, threshold)
    plt.plot(pnoderegionfrequency,pnoderegionpower)
    peaks_found=[]
    for i in best_peak_locations:
        for h in i:
            peaks_found.append(h)

    if not peaks_found:
        return 0
    else:
        closest_value = min(peaks_found, key=lambda x: abs(x - find_initial_peak()))
    target = find_other_overtone_near_numax(x,peaks_found[0], threshold)
    best_result_num_peaksbest_peak_locations,best_numax,best_starting_peak =find_best_numax_from_both_peak(
        find_initial_peak(),find_other_overtone_near_numax(
            find_initial_peak(),peaks_found[0], threshold), threshold)

    plt.figure()
    plt.plot(pnoderegionfrequency,pnoderegionpower)

    full_list_peaks=[np.argmax(pnoderegionpower),find_other_overtone_near_numax(
        find_initial_peak(),peaks_found[0], threshold)]
    for i in best_peak_locations:
        for h in i:

            full_list_peaks.append(h)
            plt.axvline(pnoderegionfrequency[h],color='red')


    plt.axvline(pnoderegionfrequency[np.argmax(pnoderegionpower)], color='green')
    plt.axvline(pnoderegionfrequency[find_other_overtone_near_numax(
        find_initial_peak(),peaks_found[0], threshold)], color='green')
    plt.axhline(threshold,color='black')
    plt.show()
    if len(full_list_peaks) != len(set(full_list_peaks)):
        return 0
    else:
        return 1


"""
The peak_locations function finds the spacing between each peak value within the found p
node by using the scipy peak finder package.
The cutoff height has been given by half of the maximum peak height however this should be
adjusted so it can include all relavent peaks within the node.
The minimum peak spacing has been given as 1/5 of the p-node range in this case however this
can be adjusted if needed to.

Returns peak_spacing, an array of the distances between each peak, and peaks, which has
the frequency and max height of each peak used, this is used for graphing only.
"""
from scipy.signal import find_peaks
import math
# Freq1 and Power1 will be the binned frequency/power over the p-node range, not the whole range
def peak_locations(freq1, power1, param2): 
    h = max(power1)*param2
    x_spacing = freq1[-1] - freq1[0]
    d = x_spacing/500
    peaks, _ = find_peaks(power1, distance=d)
    peak = freq1[peaks]
    peak_spacing = []
    for i in range(0, len(peak)-1):
        peak_spacing.append(peak[i+1]-peak[i])
    return peak_spacing, peaks


def plotpeaks(newfreq,newpower,param2):
    plt.plot(np.array(newfreq), np.array(newpower), label='P-mode envolope', lw=0.8, color = 'blue')
    z = peak_locations(newfreq, newpower, param2)
    q = z[1]
    plt.plot(newfreq[q], newpower[q], 'rx', label='Peaks')

"""
The function is_it_a_star looks at the peak data from peak_locations and determines whether
its just noise thats been found or a stars p-node. It does this by taking the 25th
and 75th percentile peak difference and looking for a percentage of the peaks to be within
the two percentile points. The number of peaks necessary can be varied to find more or less
stars with varying accuracy. To use this use the following lines of code:
v = peak_locations(freq1, power1)
peak_spacing = v[0]

"""

def is_it_a_star(peak_spacing):
    req_peaks = math.ceil(len(peak_spacing)/2)
    percentile_25 = np.percentile(peak_spacing, 25)-0.1
    percentile_75 = np.percentile(peak_spacing, 75)+0.1
    peaks_within_percentiles = [diff for diff in peak_spacing if percentile_25 < diff < percentile_75]
    if len(peaks_within_percentiles) < req_peaks:
        return 0
    else:
        return 1

def deltavSR(vmax):
  return (135 * (vmax/3000)**0.791)

def Amax(Teff, vmax):
    amaxsolar = 3
    Teffsolar = 5500
    Tred = 8907 * (0.5)**-0.093
    beta = 1 - np.exp(-(Tred - Teff)/(1550))
    return (amaxsolar * beta * (vmax/3000)**-1 * (Teff/Teffsolar)**1.5)

def getheight(dv, Amax):
  return 0.1*(135/dv)*((Amax/3)**2)

def H(vmax, Teff):
  dv = deltavSR(vmax)
  Ampmax = Amax(Teff, vmax)
  return getheight(dv, Ampmax)

def gaussian(x, height, center, fwhm):
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))
    return (height * np.exp(-(x - center)**2 / (2 * sigma**2))) + 1, sigma

def findtotalp(vmax, power, frequency):
  #for a given vmax, calculates the total power under the envelope.

  FWHM = (0.66 * (vmax)**0.88)
  sigma = FWHM/(2*np.sqrt(2*np.log(2)))

  if (vmax-(3*sigma)) > 100:
    powertobesummed = power[(frequency > (vmax-2*sigma)) & (frequency < (vmax+2*sigma))]
    return np.sum(powertobesummed), sigma
  else:
    return 0,0


def get_vmax_power(frequency, power):
  #iterate 'findtotalp' through the entire code
  sigmaarr = []
  pareas = []
  for i in tqdm(range(len(frequency))):
    vmax = frequency[i]
    tempparea,tempsigma = findtotalp(vmax = vmax, power=power, frequency=frequency)
    pareas.append(tempparea)
    sigmaarr.append(tempsigma)
  return pareas, sigmaarr


def expectedpspec(binnedfrequencies):

  Tarr = np.linspace(3500,5000,15)
  PowerSpectrum = []

  for i in range(len(Tarr)):
    temppspec = []
    for j in range(len(binnedfrequencies)):
      if binnedfrequencies[j] > 1000:
        y, sig =gaussian(binnedfrequencies,H(
            binnedfrequencies[j],Tarr[i]), binnedfrequencies[j], (0.66 * (binnedfrequencies[j])**0.88))
        temppspec.append(np.sum(y[abs(binnedfrequencies-binnedfrequencies[j]) < 3 * sig]))
      else:
        temppspec.append(0)
    PowerSpectrum.append(temppspec)

  return PowerSpectrum

def negative_exponential(x, a, b, c):
    return a * np.exp(-b * x) + c


def plotparea(binnedfrequencies, binnedpower,vmax):
  pareas, sigmaarr = get_vmax_power(binnedfrequencies,binnedpower)
  Powerspec = expectedpspec(binnedfrequencies)
  plt.plot(binnedfrequencies, pareas, label='Data Power Spectrum', lw=0.8, color = 'red')
  plt.plot(binnedfrequencies, Powerspec[0], label='Expected Power Spectrum', lw=0.8, color = 'orange')
  plt.axvline(vmax, color='blue')
  plt.xlabel('Vmax')
  plt.ylabel('P-area')
  plt.title('Expected and Measured Total P-Mode Power')
  plt.legend()
  plt.grid(True)
  plt.show()

def get_p_area_diff(binnedfrequencies,binnedpower):
  pareas, sigmaarr = get_vmax_power(binnedfrequencies,binnedpower)
  Powerspec = expectedpspec(binnedfrequencies)
  return np.abs(np.array(Powerspec[0])-np.array(pareas)), sigmaarr

def plotpdiff(binned_f,pdiff,model,vmax):
  plt.plot(np.array(binned_f), pdiff, label='Measured Power Difference', lw=0.8, color = 'red')
  plt.plot(np.array(binned_f), model, label='Modelled Power Difference', lw=0.8, color = 'orange')
  plt.axvline(vmax, color='blue')
  plt.xlabel('Vmax')
  plt.ylabel('P-area difference')
  plt.title('Difference in Measured and Expected Power Under Envelope')
  plt.legend()
  plt.grid(True)
  plt.show()

def plotpmodes(binned_f,binned_p,newfreq,newpower,threshold,pmodes,pmodes2):
  plt.plot(np.array(binned_f), np.array(binned_p)/pmodes, label='Power Spectrum', lw=1)
  plt.plot(np.array(newfreq), np.array(newpower)/pmodes2, label='P-mode envolope', lw=0.8, color = 'red')
  #plt.plot(np.array(binned_f),pmodes, color = 'orange')
  plt.axhline(y=threshold+1, color='orange')
  plt.xlabel('Frequency')
  plt.ylabel('Power')
  plt.title('Binned Power Spectrum Plot')
  plt.legend()
  plt.grid(True)
  plt.xlim(0,20000)
  plt.ylim(0,4)
  plt.show()
    #currently plotting unflattenned data

def linfunc(x,a,b):
    return (a*np.array(x) + b)

def pdiff_model(binned_f, pdiff, newfreq, sigmaarr):
    filteredfreq = np.array(binned_f)[(np.array(
        binned_f)>1000) & (np.array(binned_f)< (np.array(binned_f)[-1]-3*sigmaarr[-1]))]
    filteredpdiff = np.array(pdiff)[(np.array(
        binned_f)>1000) & (np.array(binned_f)< (np.array(binned_f)[-1]-3*sigmaarr[-1]))]
    binnedfreq_filtered = np.delete(
        filteredfreq, np.where((filteredfreq > newfreq[0]) & (filteredfreq < newfreq[-1])))
    binnedpdiff_filtered = np.delete(
        filteredpdiff, np.where((filteredfreq > newfreq[0]) & (filteredfreq < newfreq[-1])))

    initial_guess = [0.01, 5]
    params, covariance = curve_fit(linfunc, binnedfreq_filtered, binnedpdiff_filtered,p0=initial_guess)
    a_fit, b_fit = params

    model = linfunc(binned_f,a_fit,b_fit)
    return model

def findtrough(binned_f,pdiff,model, sigmaarr):
    filteredpdiff = np.array(pdiff)[(
        np.array(binned_f)>2000) & (np.array(binned_f)< (np.array(binned_f)[-1]-3*sigmaarr[-1]))]
    filteredmodel = np.array(model)[(
        np.array(binned_f)>2000) & (np.array(binned_f)< (np.array(binned_f)[-1]-3*sigmaarr[-1]))]
    difference = np.abs(filteredpdiff - filteredmodel)

    vmax = binned_f[(np.array(
        binned_f)>2000) & (np.array(
        binned_f)< (np.array(binned_f)[-1]-3*sigmaarr[-1]))][np.where(difference == max(difference))]
    return(max(difference), vmax)


def optimal_binning(freq, powerr, freqBinSize):
    maxRange = []
    freqArr = []
    frequency = np.array(freq[freq>100])
    power = np.array(powerr[freq>100])

    while freqBinSize<50:
        rangeInBinArr = []
        freqInBinArr = []
        freqbinned = []
        powerbinned = []


        binSize=0
        while (frequency[binSize]-frequency[0])<freqBinSize:
            binSize+=1


        for i in range(0,len(frequency)-binSize,binSize): 
            sum1=0
            for j in range(binSize):
                sum1+=power[i+j]
            freqbinned.append(frequency[i])
            powerbinned.append(sum1/binSize)


        binRange2 = 0
        while freqbinned[binRange2] - freqbinned[0]<freqBinSize*5:
            binRange2+=1

        for i in range(0, len(freqbinned)-binRange2, binRange2): 
            powerRange = powerbinned[i:i+binRange2]
            smallest = min(powerRange)
            largest = max(powerRange)
            rangeInBin = largest - smallest
            rangeInBinArr.append(rangeInBin)
            freqInBinArr.append(freqbinned[i])

        freqIndex = 0
        while freqInBinArr[freqIndex]>12000:
            freqIndex+=1

        noiseRange = max(np.array(rangeInBinArr)[(np.array(freqInBinArr)) > 5000]) # 5000 may need to change
        pmodeRange = max(rangeInBinArr) # Finds range of the pmode
        difference = pmodeRange - noiseRange # Finds the difference between them
        freqArr.append(freqBinSize) # Frequency of bins used to smooth data
        maxRange.append(difference) # difference between power range in pmode section and noise section
        freqBinSize+=1

    idealFreq = freqArr[maxRange.index(min(maxRange))]
    return idealFreq





def binning(frequency, power, binsize, method = "mean"):
    freqbinned = []
    powerbinned = []

    optimalBinSize=0
    while frequency[optimalBinSize]-frequency[0]<binsize:
        optimalBinSize+=1 # Size of bin to be used when binning
    for i in range(0,len(frequency)-optimalBinSize,optimalBinSize): 
        if method == "mean":
            sum1=0
            for j in range(optimalBinSize):
                sum1+=power[i+j]
            freqbinned.append(frequency[i])
            powerbinned.append(sum1/optimalBinSize)
        elif method == "median":
            temppower = power[(frequency>frequency[i])&(frequency<frequency[(i+optimalBinSize)])]
            freqbinned.append(frequency[i])
            powerbinned.append(np.median(temppower))
        else:
            print("error, bin method unspecified/unsupported")

    return np.array(freqbinned), np.array(powerbinned)


def findpregion2(binsize, binpowerarr, midpointsarr):
    #startpoint=2000
    freqrange = 1500
    numbins = int(freqrange//binsize)
    deltaparr = []
    iterationrange = int((midpointsarr[-1]-freqrange))
    indeces = np.where(np.abs(np.gradient(binpowerarr))<0.00025)


    for j in indeces[0]:
        deltap = 0
        inifreq = midpointsarr[j]
        temppower = binpowerarr[(midpointsarr > inifreq) & (midpointsarr < (inifreq + freqrange))]
        tempfreq = midpointsarr[(midpointsarr > inifreq) & (midpointsarr < (inifreq + freqrange))]

        for i in range(numbins-2):
            if i + 1 < len(temppower):
                deltap += np.abs(temppower[i+1]-temppower[i])
        deltaparr.append(deltap)

    initialfrequency = midpointsarr[indeces[0][(deltaparr.index(max(deltaparr)))]]
    finalfrequency = initialfrequency + freqrange

    fmax = (initialfrequency + finalfrequency)/2
    FWHM = (0.66 * (fmax)**0.88)
    sigma = FWHM/(2*np.sqrt(2*np.log(2)))

    newpower = binpowerarr[(midpointsarr > (fmax - 3*sigma)) & (midpointsarr < fmax + 3*sigma)]
    newfreq = midpointsarr[(midpointsarr > (fmax - 3*sigma)) & (midpointsarr < fmax + 3*sigma)]

    whitenoisepnan = binpowerarr[midpointsarr > (fmax + 4*sigma)]
    nan_mask = np.isnan(whitenoisepnan)
    whitenoisep = whitenoisepnan[~nan_mask]

    return newfreq, newpower, np.mean(whitenoisep), np.where((
        midpointsarr > (fmax - 3 * sigma)) & (midpointsarr < fmax + 3 * sigma))[0]



def newgranmodel(frequency, power, binnedf, binnedp):
    logfreq = np.log(frequency)[1:]
    power1 = power[1:]
    a = -0.005
    b = np.log(1.1)
    binlocs = np.linspace(logfreq[0], logfreq[-1], 100)

    powerbinned=[]
    print(len(logfreq))
    counter=0
    for i in binlocs:
        temp = power1[np.abs(logfreq-i) < (a * i) + b]
        tempfreq = logfreq[np.abs(logfreq-i) < (a * i) + b]
        #print(np.abs(logfreq-i) < a * i + b)
        endofbin = len(temp)//5
        temp2 = temp[:endofbin]
        powerbinned.append(np.median(temp))
        counter+=1


    #powerbinned = [np.median(power1[np.abs(logfreq - i) < a * i + b]) for i in binlocs]

    pf = interpolate.interp1d(binlocs, powerbinned, bounds_error=False)
    p2 = pf(np.log(binnedf))
    return p2




# def grandfunc(frequency, power, param1, param2, param3):

#   result1,result2,result3=(0,0,0)

#   binned_f1, binned_p1 = binning(frequency, power, 30, method = 'mean')
#   newfreq1, newpower1, threshold1, pmode_indices = findpregion(30, np.array(binned_p1), np.array(binned_f1))

#   plt.figure()
#   plt.plot(binned_f1, binned_p1, label = 'Frequency Power Spectrum',color='blue')
#   plt.plot(newfreq1, newpower1, label='P-mode region', color='red')
#   plt.xlim(0,7500)
#   plt.show()

#   binsize = optimal_binning(frequency,power,0.2)
#   print(binsize)
#   #finds optimal binsize

#   binned_f, binned_p = binning(frequency, power, binsize, method = "mean")
#   #bins the data

#   """pmodes, params = GranModel(np.delete(binned_f,np.where(np.isnan(binned_p)==True)),
#                              np.delete(binned_p,np.where(np.isnan(binned_p)==True)), newfreq1, newpower1)"""
#   #models the granulation for flattening

#   pmodes = newgranmodel(np.array(frequency), np.array(power), np.array(binned_f), np.array(binned_p))

#   newfreq2, newpower2, threshold2, pmode_indices = findpregion(
#       binsize, np.array(binned_p)/pmodes, np.array(binned_f))
#   #identifies the p-mode region

#   """pmodes2 = negative_exponential(newfreq2, params[0], params[1], params[2])"""
#   pmodes2 = newgranmodel(newfreq2, newpower2, newfreq2, newpower2)

#   plotpmodes(binned_f,binned_p,newfreq2,newpower2*pmodes2,threshold2,pmodes,pmodes2)

#   result1 = grand_function(binned_f, binned_p/pmodes, binsize, param1)
#   #method 1

#   peakspacing = peak_locations(newfreq2, newpower2/pmodes2, param2)[0]
#   plotpeaks(newfreq2, newpower2, param2)
#   result2 = is_it_a_star(peakspacing)
#   #method 2

#   pdiff,sigmaarr = get_p_area_diff(np.array(binned_f), np.array(binned_p)/pmodes)
#   model = pdiff_model(binned_f, pdiff, newfreq2, sigmaarr)
#   difference,vmax2 = findtrough(binned_f,pdiff,model, sigmaarr)
#   if difference > param3:
#     result3+=1
#   #method 3
#   plotparea(binned_f, binned_p/pmodes,vmax2)
#   plotpdiff(binned_f,pdiff,model,vmax2)

#   return result1,result2,result3
def findpregion(binsize, binpowerarr, midpointsarr):
  startpoint=1500
  freqrange = 1500
  numbins = int(freqrange//binsize)
  deltaparr = []
  iterationrange = int((midpointsarr[-1]-freqrange))

  for j in tqdm(range(0,iterationrange,100)):
    deltap = 0
    inifreq = startpoint+j
    temppower = binpowerarr[(midpointsarr > inifreq) & (midpointsarr < (inifreq + freqrange))]
    tempfreq = midpointsarr[(midpointsarr > inifreq) & (midpointsarr < (inifreq + freqrange))]

    for i in range(numbins-2):
      if i + 1 < len(temppower):
        deltap += np.abs(temppower[i+1]-temppower[i])

    deltaparr.append(deltap)

  initialfrequency = startpoint+(deltaparr.index(max(deltaparr))*100)
  finalfrequency = initialfrequency + freqrange

  fmax = (initialfrequency + finalfrequency)/2
  FWHM = (0.66 * (fmax)**0.88)
  sigma = FWHM/(2*np.sqrt(2*np.log(2)))

  newpower = binpowerarr[(midpointsarr > (fmax - 3*sigma)) & (midpointsarr < fmax + 3*sigma)]
  newfreq = midpointsarr[(midpointsarr > (fmax - 3*sigma)) & (midpointsarr < fmax + 3*sigma)]

  whitenoisepnan = binpowerarr[midpointsarr > (fmax + 4*sigma)]
  nan_mask = np.isnan(whitenoisepnan)
  whitenoisep = whitenoisepnan[~nan_mask]



  return newfreq, newpower, np.mean(whitenoisep), np.where((midpointsarr > (fmax - 3 * sigma)) & (midpointsarr < fmax + 3 * sigma))[0]


# ---------- Training Preperation ----------

In [4]:
# # Define the directory paths
# training_direct = "/Users/elicox/Desktop/Mac/Work/Yr3 Work/Group Studies/machine_learning_files/Training_data/" #folder containing the data to be used to train
# labelled_direct = "/Users/elicox/Desktop/Mac/Work/Yr3 Work/Group Studies/machine_learning_files/Labelled_data/" #folder containing the training data thats been labelled
# savemodel_file = "/Users/elicox/Desktop/Mac/Work/Yr3 Work/Group Studies/machine_learning_files/Model_saving/test_model.pkl" #file for the saved model
# figsaving_direct = "/Users/elicox/Desktop/Mac/Work/Yr3 Work/Group Studies/machine_learning_files/Figure_saving" #folder containing saved figures
# data_direct = "/Users/elicox/Desktop/Mac/Work/Yr3 Work/Group Studies/machine_learning_files/Unseen_data" #folder containing regular unseen data

In [5]:
def add_third_column(frequency, power, labels):
    """
    Adds a third column to the labels based on specified conditions.

    This function calculates a bin size using optimal binning method and identifies a region
    within the frequency range. It then updates the labels array by setting the values to 1
    where the condition is satisfied based on the calculated region.

    Parameters:
    - frequency (array-like): Array containing frequency data.
    - power (array-like): Array containing power data.
    - labels (array-like): Array of labels to be updated.

    Returns:
    - labels (array-like): Updated array of labels.
    """
    # Calculate bin size using optimal binning method
    binsize = optimal_binning(np.array(frequency), np.array(power), 0.2)
    
    # Find the frequency region within the bin size
    newfreq = findpregion2(binsize, power, frequency)[0]  # We only need the first value
    
    # Find indices where the condition is satisfied and update labels
    condition_indices = np.where((frequency >= newfreq[0]) & (frequency <= newfreq[-1]))[0]
    labels[condition_indices] = 1
    
    return labels

def split_data_sets(file_path, test_size=0.15, val_size=0.15, random_state=42):
    """
    This function splits the data from a file into training, validation, and test sets.

    Parameters:
    - file_path (str): The path to the file containing the data.
    - test_size (float): The proportion of the dataset to include in the test split (default is 0.15).
    - val_size (float): The proportion of the dataset to include in the validation split (default is 0.15).
    - random_state (int): Controls the shuffling applied to the data before splitting (default is 42).

    Returns:
    - X_train (array-like): The feature matrix for the training set.
    - X_val (array-like): The feature matrix for the validation set.
    - X_test (array-like): The feature matrix for the test set.
    - y_train (array-like): The labels for the training set.
    - y_val (array-like): The labels for the validation set.
    - y_test (array-like): The labels for the test set.
    """
    data = np.genfromtxt(file_path, delimiter=' ', skip_header=2)

    # Extract binned_f, normp, and labels
    binned_f = data[:, 0]
    normp = data[:, 1]
    labels = data[:, 2]  # Assuming the third column contains labels

    # Combine binned_f and normp into feature array
    X = np.column_stack((binned_f, normp))

    # Split the data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, labels,
                                                            test_size=test_size, random_state=random_state)

    # Further split the training data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train,
                                                          test_size=val_size, random_state=random_state)

    return X_train, X_val, X_test, y_train, y_val, y_test

In [6]:
#fn to split files from tsm group into smaller files
"""
def split_csv_files(input_folder, output_folder):
    # Iterate over all files in the input folder
    for filename in tqdm(os.listdir(input_folder), desc="Processing files"):
        if filename.endswith('.csv'):
            file_path = os.path.join(input_folder, filename)
            df = pd.read_csv(file_path)
            
            # Create a new DataFrame with the entire second column (frequency)
            frequency_df = pd.DataFrame(df.iloc[:, 2])  # assuming frequency is in the 3rd column
            
            # Iterate through power spectrum columns
            for col_idx in range(3, df.shape[1]): #4th column onwards
                spectrum_col = df.columns[col_idx]
                
                # Create a new DataFrame combining frequency and current power spectrum column
                combined_df = pd.concat([frequency_df, df[spectrum_col]], axis=1)
                
                # Save the DataFrame to a new CSV file with space as delimiter
                output_filename = f"{os.path.splitext(filename)[0]}_{spectrum_col}.txt"
                output_path = os.path.join(output_folder, output_filename)
                combined_df.to_csv(output_path, sep=' ', index=False)

# Input and output directory paths for the new folder 'first15'
input_directory_first15 = "/Users/elicox/Desktop/Mac/Work/Yr3 Work/Group Studies/machine_learning_files/first15/"
output_directory_first15 = "/Users/elicox/Desktop/Mac/Work/Yr3 Work/Group Studies/machine_learning_files/first15_files/"

# Execute the function for the 'first15' folder
split_csv_files(input_directory_first15, output_directory_first15)

"""

'\ndef split_csv_files(input_folder, output_folder):\n    # Iterate over all files in the input folder\n    for filename in tqdm(os.listdir(input_folder), desc="Processing files"):\n        if filename.endswith(\'.csv\'):\n            file_path = os.path.join(input_folder, filename)\n            df = pd.read_csv(file_path)\n            \n            # Create a new DataFrame with the entire second column (frequency)\n            frequency_df = pd.DataFrame(df.iloc[:, 2])  # assuming frequency is in the 3rd column\n            \n            # Iterate through power spectrum columns\n            for col_idx in range(3, df.shape[1]): #4th column onwards\n                spectrum_col = df.columns[col_idx]\n                \n                # Create a new DataFrame combining frequency and current power spectrum column\n                combined_df = pd.concat([frequency_df, df[spectrum_col]], axis=1)\n                \n                # Save the DataFrame to a new CSV file with space as delimi

## ---------- Generate The Training Files ----------

In [7]:
# code to process the data
"""
# Get a list of all files in the input directory
file_names = os.listdir(training_direct)

# Exclude system files like .DS_Store
file_names = [file for file in os.listdir(training_direct) if file != '.DS_Store']
# Iterate over each file in the input directory
for file_name in tqdm(file_names, desc='Processing files'):
    # Load the data
    data = np.genfromtxt(os.path.join(training_direct, file_name), skip_header=2)# skip 2 still gives nan
    frequency = data[:, 0]
    power = data[:, 1]
    
    binsize = optimal_binning(np.array(frequency), np.array(power), 0.2)
    binned_f, binned_p = binning(frequency, power, binsize, method = "mean")
    pmodes = newgranmodel(frequency, power, binned_f, binned_p)
    normp = binned_p / pmodes
    
    # Apply add_third_column function
    labels = np.zeros_like(binned_f)  # Initialize labels with zeros
    labels = add_third_column(binned_f, normp, labels)

    # Define the new file name
    labelled_file_name = "processed_" + file_name

    # Save the processed data into a new file with three columns
    np.savetxt(
        os.path.join(labelled_direct, labelled_file_name),
        np.column_stack((binned_f, normp, labels)),
        delimiter=" ",
        fmt=['%.18e', '%.18e', '%d']
    )
"""

'\n# Get a list of all files in the input directory\nfile_names = os.listdir(training_direct)\n\n# Exclude system files like .DS_Store\nfile_names = [file for file in os.listdir(training_direct) if file != \'.DS_Store\']\n# Iterate over each file in the input directory\nfor file_name in tqdm(file_names, desc=\'Processing files\'):\n    # Load the data\n    data = np.genfromtxt(os.path.join(training_direct, file_name), skip_header=2)# skip 2 still gives nan\n    frequency = data[:, 0]\n    power = data[:, 1]\n    \n\n#     # Apply grandfunc\n#     binned_f, binned_p from binned, pmodes=p2 = grandfunc(frequency, power)\n#     normp = binned_p / pmodes\n    binsize = optimal_binning(np.array(frequency), np.array(power), 0.2)\n    binned_f, binned_p = binning(frequency, power, binsize, method = "mean")\n    pmodes = newgranmodel(frequency, power, binned_f, binned_p)\n    normp = binned_p / pmodes\n    \n    # Apply add_third_column function\n    labels = np.zeros_like(binned_f)  # Initiali

# ---------- Machine Learning ----------

## ---------- MODEL RESET ----------

In [8]:

# ONLY RUN IF THE MACHINE BECOMES OVERTRAINED
def reset_arf_model():
    """
    Reset the ARF model to its initial state by reinitializing a new ARF model.
    
    Returns:
    - arf_model: The reset ARF model.
    """
    arf_model = ensemble.AdaptiveRandomForestClassifier()
    return arf_model
arf_model = reset_arf_model()

## ---------- Machine Learning Code ----------

In [9]:
def train_model(X_train, y_train, X_val, y_val, white_noise, num_iterations=100, batch_size=32, \
                early_stopping_rounds=10):
    arf_models = []
    best_f1_score = -float('inf')
    best_threshold_multiplier = None
    best_arf_model = None
    
    # Shuffle the training data once
    shuffled_indices = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[shuffled_indices]
    y_train_shuffled = y_train[shuffled_indices]
    
    # Iterate through a range of threshold multiplier values
    threshold_multipliers = np.linspace(0.01, 5.0, num=20)  # Adjust the range as needed
    for threshold_multiplier in tqdm(threshold_multipliers, desc='Threshold Multipliers'):
        arf_model = AdaptiveRandomForestClassifier()
        best_val_f1 = -float('inf')
        rounds_since_best = 0
        first_non_zero_f1 = False  # Flag to indicate if there has been at least one non-zero F1 score
        
        # Training in batches
        for iteration in range(num_iterations):
            for batch_start in range(0, len(X_train), batch_size):
                batch_end = min(batch_start + batch_size, len(X_train))
                X_batch = X_train_shuffled[batch_start:batch_end]
                y_batch = y_train_shuffled[batch_start:batch_end]
                
                arf_model.partial_fit(X_batch, y_batch, classes=[0, 1])
                
            val_probs = arf_model.predict_proba(X_val)[:, 1]
            threshold = white_noise * threshold_multiplier
            val_f1 = f1_score(y_val, (val_probs > threshold).astype(int))
            
            if val_f1 >= best_val_f1:
                best_val_f1 = val_f1
                rounds_since_best = 0
                if val_f1 > 0:  # Check if the F1 score is non-zero
                    first_non_zero_f1 = True
            else:
                rounds_since_best += 1
                if rounds_since_best >= early_stopping_rounds or (val_f1 == 0 and first_non_zero_f1):
                    break
        print(f"Threshold Multiplier:{threshold_multiplier},Threshold:{threshold},Val F1 Score:{best_val_f1}")
        arf_models.append((arf_model, best_val_f1))  # Store the model along with its F1 score
    
    # Sort the list of models and F1 scores based on the F1 score in ascending order
    sorted_models = sorted(arf_models, key=lambda x: x[1])
    
    # Find the best model based on the highest F1 score
    best_arf_model, best_f1_score = sorted_models[-1]  # Get the model with the highest F1 score
    best_threshold_multiplier = threshold_multipliers[np.argmax([x[1] for x in arf_models])]
    
    return best_arf_model, best_threshold_multiplier

def test_model(model, X_test, y_test, threshold_multiplier, white_noise):
    """
    Test a trained model on a separate test dataset.

    Parameters:
    - model: The trained model to be tested.
    - X_test (array-like): The feature matrix for the test data.
    - y_test (array-like): The labels for the test data.
    - threshold_multiplier (float): The threshold multiplier determined during evaluation.
    - white_noise (float): The level of white noise used to calculate the threshold.

    Returns:
    - test_accuracy (float): The accuracy of the model on the test data.
    - test_f1_score (float): The F1 score of the model on the test data.
    - confusion_mat (ndarray): The confusion matrix of the model on the test data.
    """
    # Calculate the threshold using the threshold multiplier and white noise level
    threshold = threshold_multiplier * white_noise
    
    # Predict probabilities for the test data and convert to binary predictions using the threshold
    y_pred_test = (model.predict_proba(X_test)[:, 1] > threshold).astype(int)
    
    # Calculate accuracy and F1 score for the test data
    test_accuracy = accuracy_score(y_test, y_pred_test)
    test_f1_score = f1_score(y_test, y_pred_test)
    
    # Compute the confusion matrix for the test data
    confusion_mat = confusion_matrix(y_test, y_pred_test)
    
    return test_accuracy, test_f1_score, confusion_mat

def train_and_save_model(X_train, y_train, X_val, y_val, save_path, num_iterations=10):
    """
    Train a model on the given data and save the trained model to a file.

    Parameters:
    - X_train (array-like): The feature matrix for the training data.
    - y_train (array-like): The labels for the training data.
    - X_val (array-like): The feature matrix for the validation data.
    - y_val (array-like): The labels for the validation data.
    - save_path (str): The file path where the trained model will be saved.
    - num_iterations (int): The number of iterations for which to train the model (default is 10).

    Returns:
    - arf_model: The trained model that has been saved.
    """
    # Train the model
    arf_model = train_model(X_train, y_train, X_val, y_val, num_iterations=num_iterations)
    
    # Save the trained model
    joblib.dump(arf_model, save_path)
    print("Model saved successfully!")
    
    return arf_model  # Return the trained model
# 

def plot_confusion_matrix(TP,TN,FP,FN,accu_percent,train_time,threshold_multiplier,file_name,figsaving_direct):
    """
    Plot a confusion matrix with True Positives (TP), True Negatives (TN),
    False Positives (FP), and False Negatives (FN), along with the accuracy percentage, 
    training time, and threshold multiplier.

    Parameters:
    - TP (int): True Positives.
    - TN (int): True Negatives.
    - FP (int): False Positives.
    - FN (int): False Negatives.
    - accu_percent (float): The accuracy percentage of the model.
    - train_time (float): The training time in seconds.
    - threshold_multiplier (float): The best threshold multiplier.
    - file_name (str): The name of the file being processed.
    - figsaving_direct (str): The directory where the figures will be saved.
    """
    confusion_mat = np.array([[TP, FN],
                               [FP, TN]])  # Adjust the arrangement of confusion matrix elements here
    
    plt.figure(figsize=(12, 10))  # Increased figure size for presentation
    sns.set(font_scale=1.5)
    sns.heatmap(confusion_mat, annot=True, cmap='Blues', fmt='g',
                xticklabels=['Positive', 'Negative'],
                yticklabels=['Positive', 'Negative'], cbar_kws={'label': 'Count'},
                annot_kws={'size': 16})

    # Font size adjustments
    plt.xlabel('Predicted Labels', fontsize=18)
    plt.ylabel('True Labels', fontsize=18)
    title_text = f'Confusion Matrix for {file_name}\nAccuracy: {accu_percent:.2f}%'

    if train_time != 0:
        title_text += f', Training Time: {train_time:.2f}s'

    if threshold_multiplier != 0:
        title_text += f', Threshold Multiplier: {threshold_multiplier:.2f}'

    plt.title(title_text, fontsize=20, fontweight='bold')  # Adjust fontweight and color
    # Define annotation text
    annotation_text = f"True Positives: {TP} True Negatives: {TN} False Positives: {FP} False Negatives: {FN}"

    # Print annotation underneath x-axis label
    plt.text(0.5,-0.11,annotation_text,horizontalalignment='center',transform=plt.gca().transAxes,fontsize=16)

    fig_path = os.path.join(figsaving_direct, file_name + '_confusion_matrix.png')
    plt.savefig(fig_path, dpi=300)
    plt.close()


def process_file(file_names, labelled_direct, savemodel_direct, figsaving_direct):
    best_models = {}  # Dictionary to store the best models for each file
    
    results = []
    
    for filename in tqdm(file_names, desc='Processing files'):
        print(filename)
        file_path = os.path.join(labelled_direct, filename)
        
        # Load the data from the file
        data = np.genfromtxt(file_path, skip_header=2)
        frequency = data[:, 0]
        power = data[:, 1]
        
        # Calculate bin size and white noise using all of the data
        binsize = optimal_binning(frequency, power, 0.2)
        white_noise = findpregion(binsize, power, frequency)[2]

        # Split data into training, validation, and test sets
        X_train, X_val, X_test, y_train, y_val, y_test = split_data_sets(file_path)
        
        # Measure the start time for training
        start_time = time.time()
        
        # Train the model
        best_arf_model, threshold_multiplier = train_model(X_train, y_train, X_val, y_val, white_noise)
        
        # Test the model
        test_accuracy, test_f1_score, confusion_mat = test_model(best_arf_model, X_test, y_test, \
                                                                 threshold_multiplier, white_noise)
        
        # Save the model if it's the best for this file
        if filename not in best_models or test_f1_score > best_models[filename]['test_f1_score']:
            best_models[filename] = {
                'model': best_arf_model,
                'threshold_multiplier': threshold_multiplier,
                'test_accuracy': test_accuracy,
                'test_f1_score': test_f1_score,
                'confusion_matrix': confusion_mat
            }
            # Save the model to disk
            model_save_path = os.path.join(savemodel_direct, f"best_model_{filename}.joblib")
            joblib.dump(best_arf_model, model_save_path)
            print(f"Best model saved for {filename} with F1 score: {test_f1_score}")

        # Measure the end time for training
        end_time = time.time()
        train_time = end_time - start_time
        
        # Calculate TP, TN, FP, FN
        TP = confusion_mat[1, 1]
        TN = confusion_mat[0, 0]
        FP = confusion_mat[0, 1]
        FN = confusion_mat[1, 0]
        
        # Calculate accuracy percentage
        accu_percent = (TP + TN) / (TP + TN + FP + FN) * 100
        
        # Plot confusion matrix with metrics and save the figure
        plot_confusion_matrix(TP, TN, FP, FN, accu_percent, train_time, threshold_multiplier, filename, \
                              figsaving_direct)
        
        # Append results
        results.append((filename, threshold_multiplier, test_accuracy, test_f1_score, confusion_mat,\
                        TP, TN, FP, FN, accu_percent, train_time))
    
    return best_models, results

def voting_ensemble(models, X_test, white_noise, threshold_multipliers):
    predictions = []
    for model, threshold_multiplier in zip(models, threshold_multipliers):
        val_probs = model.predict_proba(X_test)[:, 1]
        threshold = white_noise * threshold_multiplier
        predictions.append((val_probs > threshold).astype(int))
    
    # Transpose predictions to have each model's predictions in columns
    predictions = np.array(predictions).T
    
    # Perform majority voting
    ensemble_predictions = []
    majority_threshold_multipliers = []
    for preds in predictions:
        counter = Counter(preds)
        # Find the most common prediction(s)
        most_common_prediction = counter.most_common(1)[0][0]
        majority_predictions = [pred for pred in preds if pred == most_common_prediction]
        ensemble_predictions.append(most_common_prediction)
        # Find the threshold multiplier(s) corresponding to the most common prediction(s)
        majority_threshold_multipliers.extend([threshold_multipliers[i] for i, \
                                               pred in enumerate(preds) if pred == most_common_prediction])
    
    # Choose the most common threshold multiplier among the predictions
    ensemble_threshold_multiplier = Counter(majority_threshold_multipliers).most_common(1)[0][0]
    
    return ensemble_predictions, ensemble_threshold_multiplier


### Code to Train the Machine

In [18]:
#Code to train the machine
"""
# Get a list of all files in the labelled directory excluding .DS_Store files
file_names = [file for file in os.listdir(labelled_direct) if file != '.DS_Store']
print("File names in the directory:", file_names)  # Add this line to print the file names
arf_model = reset_arf_model() # resets the model
# Initialize a list to store trained models
trained_models = []

# Process all files together
best_models, results = process_file(file_names, labelled_direct, savemodel_direct, figsaving_direct)

# Print results for each file
for result in results:
    file_name, threshold_multiplier, test_accuracy, \
    test_f1_score, confusion_mat, TP, TN, FP, FN, \
    accu_percent, train_time = result

    print(f"File: {file_name}")
    print("Best Threshold Multiplier:", threshold_multiplier)
    print("Test Set Accuracy:", test_accuracy)
    print("Test Set F1 Score:", test_f1_score)
    print("True Positives:", TP)
    print("True Negatives:", TN)
    print("False Positives:", FP)
    print("False Negatives:", FN)
    print("Accuracy Percentage: {:.2f}%".format(accu_percent))
    print("Training Time: {:.2f} seconds".format(train_time))

    # Append the best model to the list of trained models
    trained_model_path = os.path.join(savemodel_direct, f"best_model_{file_name}.joblib")
    trained_models.append(joblib.load(trained_model_path))
"""
# Load the trained models
trained_models = []
for file_name in os.listdir(savemodel_direct):
    if file_name.endswith('.joblib'):
        model_path = os.path.join(savemodel_direct, file_name)
        trained_model = joblib.load(model_path)
        trained_models.append({'model': trained_model, 'file_name': file_name})

best_models = trained_models
# Define the directory containing the new test file
new_test_file_path = "/Users/elicox/Desktop/processed_kplr008006161_kasoc-ts_slc_v1.pow"  

# Load the data from the file
data = np.genfromtxt(new_test_file_path, skip_header=2)
frequency = data[:, 0]
power = data[:, 1]
        
        # Calculate bin size and white noise using all of the data
binsize = optimal_binning(frequency, power, 0.2)
white_noise = findpregion(binsize, power, frequency)[2]


# Split the new test data into training, validation, and test sets
X_train_new, X_val_new, X_test_new, y_train_new, y_val_new, y_test_new = split_data_sets(new_test_file_path)

# Extracting the best models from best_models
models = [best_models[file]['model'] for file in best_models]
threshold_multipliers = {file: best_models[file]['threshold_multiplier'] for file in best_models}
threshold_multipliers_list = list(threshold_multipliers.values())
# Create the ensemble predictions
ensemble_predictions = voting_ensemble(models, X_test_new, white_noise, threshold_multipliers_list)

# Get the ensemble threshold multiplier
ensemble_threshold_multiplier = np.mean(list(threshold_multipliers.values()))+1

# print("Ensemble Predictions:", ensemble_predictions)
print("Ensemble Threshold Multiplier:", ensemble_threshold_multiplier)

# Convert ensemble predictions to the same data type as y_test_new
ensemble_predictions_binary = np.array(ensemble_predictions[0], dtype=np.float64)

# Calculate confusion matrix from ensemble predictions
ensemble_confusion_mat = confusion_matrix(y_test_new, ensemble_predictions_binary)
print("Ensemble Predictions:", ensemble_predictions[0])
print("test labelles:",y_test_new)

# Unpack the elements of the confusion matrix
TP_ensemble = ensemble_confusion_mat[0, 0]
TN_ensemble = ensemble_confusion_mat[1, 1]
FP_ensemble = ensemble_confusion_mat[0, 1]
FN_ensemble = ensemble_confusion_mat[1, 0]
print("TP_ensemble",TP_ensemble)
print("TN_ensemble",TN_ensemble)
print("FP_ensemble",FP_ensemble)
print("FN_ensemble",FN_ensemble)
print("white noise",white_noise)



# Plot the confusion matrix
plot_confusion_matrix(TP_ensemble, TN_ensemble, FP_ensemble, FN_ensemble, \
                      accu_percent, train_time, ensemble_threshold_multiplier, "Ensemble", figsaving_direct)

# Load the trained models
trained_models = []
for file_info in best_models:
    trained_model = file_info['model']
    trained_models.append(trained_model)

# Define the directory containing the new test file
new_test_file_path = "/Users/elicox/Desktop/processed_kplr008006161_kasoc-ts_slc_v1.pow"  

# Load the data from the file
data = np.genfromtxt(new_test_file_path, skip_header=2)
frequency = data[:, 0]
power = data[:, 1]
        
# Calculate bin size and white noise using all of the data
binsize = optimal_binning(frequency, power, 0.2)
white_noise = findpregion(binsize, power, frequency)[2]

# Split the new test data into training, validation, and test sets
X_train_new, X_val_new, X_test_new, y_train_new, y_val_new, y_test_new = split_data_sets(new_test_file_path)

# Extracting the best models from best_models
models = trained_models
threshold_multipliers={file_info['file_name']: file_info['threshold_multiplier'] for file_info in best_models}
threshold_multipliers_list = list(threshold_multipliers.values())

# Create the ensemble predictions
ensemble_predictions = voting_ensemble(models, X_test_new, white_noise, threshold_multipliers_list)

# Get the ensemble threshold multiplier
ensemble_threshold_multiplier = np.mean(list(threshold_multipliers.values())) + 1

# Convert ensemble predictions to the same data type as y_test_new
ensemble_predictions_binary = np.array(ensemble_predictions[0], dtype=np.float64)

# Calculate confusion matrix from ensemble predictions
ensemble_confusion_mat = confusion_matrix(y_test_new, ensemble_predictions_binary)

# Unpack the elements of the confusion matrix
TP_ensemble = ensemble_confusion_mat[0, 0]
TN_ensemble = ensemble_confusion_mat[1, 1]
FP_ensemble = ensemble_confusion_mat[0, 1]
FN_ensemble = ensemble_confusion_mat[1, 0]

# Plot the confusion matrix
plot_confusion_matrix(TP_ensemble, TN_ensemble, FP_ensemble, FN_ensemble, \
                      accu_percent, train_time, ensemble_threshold_multiplier, "Ensemble", figsaving_direct)


100%|██████████| 70/70 [00:00<00:00, 1494.20it/s]


TypeError: list indices must be integers or slices, not dict